In [ ]:
try:
    from src.apple_mlx_loader import maybe_generate_with_mlx
except Exception:
    maybe_generate_with_mlx = None

# Hook MLX path into local generation
# This assumes a function `generate_local(model_name: str, prompt: str)` exists later.
# We wrap it to early-exit via MLX on Apple Silicon for Qwen2.5-3B.

def _wrap_generate_local_with_mlx():
    import inspect
    global generate_local

    if 'generate_local' not in globals():
        return  # defined later; the user can re-run this cell after definitions

    original_generate_local = generate_local

    def generate_local(model_name: str, prompt: str) -> str:  # type: ignore[no-redef]
        if maybe_generate_with_mlx is not None:
            # Map cache keys to HF ids similarly to the notebook's HF_MODEL_MAP
            HF_MODEL_MAP = {
                "google_gemma-2-2b-it": "google/gemma-2-2b-it",
                "Qwen_Qwen2.5-1.5B-Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
                "Qwen_Qwen2.5-3B-Instruct": "Qwen/Qwen2.5-3B-Instruct",
            }
            hf_model_name = HF_MODEL_MAP.get(model_name)
            if hf_model_name:
                maybe_text = maybe_generate_with_mlx(hf_model_name, prompt, max_new_tokens=20)
                if maybe_text is not None:
                    return maybe_text
        # Fallback to original path
        return original_generate_local(model_name, prompt)

    # Keep a handle to the original (optional)
    generate_local._original = original_generate_local  # type: ignore[attr-defined]

# Call the wrapper once definitions are in place; if not yet, user can call it later
_wrap_generate_local_with_mlx()


# CoT Sensitivity Experiments

This notebook tests model sensitivity to Chain-of-Thought (CoT) modifications by:
1. Selecting 100 correct examples per model/dataset combination
2. Replacing the CoT with either ellipses or incorrect reasoning
3. Measuring how often the model changes its answer

In [1]:
import os
import pickle
import json
import re
import random
import requests
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
from dotenv import load_dotenv

# Import here to avoid loading if not needed
import os
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load environment variables
load_dotenv()

# Set random seed for reproducibility
random.seed(42)
np.random.seed(42)

/Users/kyle/Documents/ws/post-hoc-reasoning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from openai import OpenAI

openai_api_key = os.environ.get("OPENAI_API_KEY")
if not openai_api_key:
    raise RuntimeError("Missing OPENAI_API_KEY in environment")

client = OpenAI(api_key=openai_api_key)


## Configuration

## Fireworks AI Setup

This notebook uses Fireworks AI for model generation through their OpenAI-compatible API.

### Prerequisites:
1. Create a Fireworks AI account at https://fireworks.ai/
2. Get your API key from the Fireworks dashboard
3. Add your API key to the `.env` file:
   ```
   FIREWORKS_API_KEY=your_api_key_here
   ```

### Model Mapping:
The `generate` function includes a `MODEL_NAME_MAP` dictionary that maps local model names (from the cache) to Fireworks model identifiers. You'll need to populate this mapping with the appropriate Fireworks model names for the models you want to test.

Example:
```python
MODEL_NAME_MAP = {
    "google_gemma-2-2b-it": "accounts/fireworks/models/gemma2-2b-it",
    "Qwen_Qwen2.5-7B-Instruct": "accounts/fireworks/models/qwen2p5-7b-instruct",
    # Add more mappings as needed
}
```

In [3]:
MODEL_NAME_MAP = {
    "google_gemma-2-2b-it": "accounts/fireworks/models/gemma2-2b-it",
    "Qwen_Qwen2.5-7B-Instruct": "accounts/fireworks/models/qwen2p5-7b-instruct",
    # Add more mappings as needed
}

In [4]:
# Path to the cache directory
CACHE_DIR = Path("final_cache/cache/experiments")

# Number of examples to sample per model/dataset
N_SAMPLES = 100

# Ellipses replacement for CoT
ELLIPSES_COT = " ".join(["..."] * 1)

## Data Loading Functions

In [5]:
def get_model_dataset_combinations():
    """Get all available model/dataset combinations from the cache."""
    combinations = []
    
    if not CACHE_DIR.exists():
        raise ValueError(f"Cache directory not found: {CACHE_DIR}")
    
    for model_dir in CACHE_DIR.iterdir():
        if not model_dir.is_dir():
            continue
        
        model_name = model_dir.name
        
        for dataset_dir in model_dir.iterdir():
            if not dataset_dir.is_dir():
                continue
            
            dataset_name = dataset_dir.name
            combinations.append((model_name, dataset_name))
    
    return sorted(combinations)

In [ ]:
# Re-wrap after definitions (safe to run multiple times)
try:
    _wrap_generate_local_with_mlx()
except NameError:
    pass


In [6]:
def load_test_generations(model_name: str, dataset_name: str) -> List[Dict]:
    """Load test generation results for a model/dataset combination."""
    model_dir = CACHE_DIR / model_name / dataset_name
    
    # Find the split directory (should be only one)
    split_dirs = list(model_dir.glob("split_*"))
    if not split_dirs:
        raise ValueError(f"No split directory found for {model_name}/{dataset_name}")
    
    split_dir = split_dirs[0]
    
    # Find the experiment hash directory
    hash_dirs = [d for d in split_dir.iterdir() if d.is_dir() and len(d.name) > 10]
    if not hash_dirs:
        raise ValueError(f"No experiment directory found for {model_name}/{dataset_name}")
    
    hash_dir = hash_dirs[0]
    
    # Load test generations
    test_gen_file = hash_dir / "data" / "test_generations.pkl"
    if not test_gen_file.exists():
        raise ValueError(f"Test generations file not found: {test_gen_file}")
    
    with open(test_gen_file, 'rb') as f:
        data = pickle.load(f)
    
    return data

In [7]:
def load_steering_results(model_name: str, dataset_name: str, alpha: int = 0) -> List[Dict]:
    """Load steering results to get full generations with CoT."""
    model_dir = CACHE_DIR / model_name / dataset_name
    
    # Find the split directory
    split_dirs = list(model_dir.glob("split_*"))
    if not split_dirs:
        return None
    
    split_dir = split_dirs[0]
    
    # Find the experiment hash directory
    hash_dirs = [d for d in split_dir.iterdir() if d.is_dir() and len(d.name) > 10]
    if not hash_dirs:
        return None
    
    hash_dir = hash_dirs[0]
    
    # Load steering results for alpha=0 (unsteered)
    steering_file = hash_dir / "steering" / f"steering_alpha_{alpha}_no.pkl"
    if not steering_file.exists():
        steering_file = hash_dir / "steering" / f"steering_alpha_{alpha}_yes.pkl"
    
    if not steering_file.exists():
        return None
    
    with open(steering_file, 'rb') as f:
        data = pickle.load(f)
    
    return data

In [8]:
def sample_correct_examples(model_name: str, dataset_name: str, n_samples: int = 100) -> List[Dict]:
    """Sample n correct examples from test generations."""
    # Load steering results to get full generations
    steering_data = load_steering_results(model_name, dataset_name, alpha=0)
    
    len_steering_data = len(steering_data)
    
    np.random.seed(42)
    rand_indices = np.random.randint(0, len_steering_data, n_samples)
    
    steering_data_sample = [steering_data[i] for i in rand_indices]
    
    # Filter for correct predictions
    correct_examples = []
    for i, item in enumerate(steering_data_sample):
        # if item['pred_letter'] == item['correct_letter']:
        if True:
            example = {
                'index': i,
                'original_prompt': item['original_prompt'],
                'generation': item['steered_generation'],
                'correct_letter': item['original_letter'], # for steering, we got it right
                'correct_answer': item['original_answer'], # for steering, we got it right
                'pred_letter_1': item['original_letter'], # first iter was correct generation
                'pred_answer_1': item['original_answer'], # first iter was correct generation
                'pred_letter_2': item['new_letter'], # second iter was steered generation (alpha=0)
                'pred_answer_2': item['new_answer'], # second iter was steered generation (alpha=0)
            }
            
            correct_examples.append(example)
    
    # Sample n examples
    if len(correct_examples) >= n_samples:
        return random.sample(correct_examples, n_samples)
    else:
        print(f"Warning: Only {len(correct_examples)} correct examples found for {model_name}/{dataset_name}")
        return correct_examples

## CoT Extraction and Replacement Functions

In [9]:
def extract_cot_from_prompt(prompt: str, generation: str) -> Tuple[str, str, str]:
    """
    Extract the CoT from a prompt.
    Returns: (prefix_before_cot, cot_content, suffix_after_cot)
    """
    prompt = prompt + " " + generation
    
    # Look for "Let's think step by step:" pattern
    cot_start_pattern = r"Let's think step by step:"
    
    # Find the last occurrence (the actual question, not examples)
    matches = list(re.finditer(cot_start_pattern, prompt))
    if not matches:
        return prompt, "", ""
    
    last_match = matches[-1]
    start_idx = last_match.end()
    
    # Find the end of CoT (before "So, the best answer is" or similar)
    end_patterns = [
        r"So,? the best answer is",
        r"Therefore,? the best answer is",
        r"The best answer is",
        r"So the answer is",
        r"So the best answer is",
        r"so the best answer is",
        r"The best answer is",
        r"the best answer is",
    ]
    
    end_idx = len(prompt)
    for pattern in end_patterns:
        match = re.search(pattern, prompt[start_idx:], re.IGNORECASE)
        if match:
            end_idx = start_idx + match.start()
            break
    
    prefix = prompt[:last_match.end()]
    cot = prompt[start_idx:end_idx].strip()
    suffix = prompt[end_idx:] if end_idx < len(prompt) else ""
    
    return prefix, cot, suffix

In [10]:
def replace_cot_with_ellipses(prompt: str) -> str:
    """
    Replace the CoT in the prompt with ellipses.
    """
    
    # Replace CoT with ellipses
    new_prompt = prompt + " " + ELLIPSES_COT + " So the best answer is:"
    
    return new_prompt

In [11]:
def generate(model_name: str, prompt: str) -> str:
    """
    Generate a response using OpenRouter completions API.
    
    Args:
        model_name: Name of the model (from cache directory)
        prompt: The input prompt
    
    Returns:
        Generated text response
    """
    # Model name mapping from cache names to OpenRouter model identifiers
    MODEL_NAME_MAP = {
        "google_gemma-2-9b-it": "google/gemma-2-9b-it",
        "Qwen_Qwen2.5-7B-Instruct": "qwen/qwen-2.5-7b-instruct",
    }
    
    # Get OpenRouter model name
    openrouter_model = MODEL_NAME_MAP.get(model_name)
    if not openrouter_model:
        raise ValueError(f"Model {model_name} not mapped to OpenRouter")
    
    # Prepare API request for OpenRouter completions endpoint
    url = "https://openrouter.ai/api/v1/completions"
    
    payload = {
        "model": openrouter_model,
        "prompt": prompt,
        "temperature": 0.7,
        "max_tokens": 150,
    }
    
    headers = {
        "Authorization": f"Bearer {os.environ.get('OPENROUTER_API_KEY')}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:8888",
        "X-Title": "CoT Sensitivity Experiments"
    }
    
    # Make API request
    response = requests.post(url, headers=headers, data=json.dumps(payload))
    response.raise_for_status()  # Raises an exception for bad status codes
    
    result = response.json()
    return result['choices'][0]['text']

In [12]:
api_key = os.environ.get("FIREWORKS_API_KEY")

In [13]:
HF_MODEL_MAP = {
    # Smaller models suitable for Mac
    "google_gemma-2-2b-it": "google/gemma-2-2b-it",
    "Qwen_Qwen2.5-1.5B-Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
    "Qwen_Qwen2.5-3B-Instruct": "Qwen/Qwen2.5-3B-Instruct"
}

In [14]:
def generate_local_batch(model_name: str, prompts: list[str], batch_size: int = 4, max_new_tokens: int = 20) -> list[str]:
    """
    Batched local generation for small HF models using the existing generate_local cache.
    - Uses greedy decoding (do_sample=False) for speed/stability
    - Decodes only newly generated tokens per sample
    - Processes prompts in chunks of `batch_size`
    """
    from transformers import AutoModelForCausalLM, AutoTokenizer
    import torch, os

    HF_MODEL_MAP = {
        "google_gemma-2-2b-it": "google/gemma-2-2b-it",
        "Qwen_Qwen2.5-1.5B-Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
        "Qwen_Qwen2.5-3B-Instruct": "Qwen/Qwen2.5-3B-Instruct",
        "microsoft_Phi-3-mini-4k-instruct": "microsoft/Phi-3-mini-4k-instruct",
        "deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    }

    hf_model_name = HF_MODEL_MAP.get(model_name)
    if hf_model_name is None:
        raise ValueError(f"Model {model_name} not mapped to Hugging Face")

    # Reuse or load model/tokenizer
    if hasattr(generate_local, "model_cache") and hf_model_name in generate_local.model_cache:
        model, tokenizer = generate_local.model_cache[hf_model_name]
    else:
        device = "mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
        tokenizer = AutoTokenizer.from_pretrained(hf_model_name, token=os.environ.get("HF_TOKEN"), trust_remote_code=True)
        model = AutoModelForCausalLM.from_pretrained(
            hf_model_name,
            token=os.environ.get("HF_TOKEN"),
            torch_dtype=torch.float16 if device != "cpu" else torch.float16,
            device_map="auto",
            # trust_remote_code=True,
            low_cpu_mem_usage=True,
        )
        if not hasattr(generate_local, "model_cache"):
            generate_local.model_cache = {}
        generate_local.model_cache[hf_model_name] = (model, tokenizer)

    model.eval()

    outputs_all: list[str] = []
    pad_id = tokenizer.eos_token_id

    # Chunk prompts into batches
    for start in range(0, len(prompts), batch_size):
        chunk = prompts[start:start + batch_size]
        enc = tokenizer(chunk, return_tensors="pt", padding=True, truncation=True, max_length=2048)
        input_lens = enc["attention_mask"].sum(dim=1)
        enc = {k: v.to(model.device) for k, v in enc.items()}

        with torch.inference_mode():
            seq = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=pad_id,
            )

        for i in range(seq.size(0)):
            new_tokens = seq[i, input_lens[i]:]
            outputs_all.append(tokenizer.decode(new_tokens, skip_special_tokens=True))

    return outputs_all


In [15]:
def generate_local(model_name: str, prompt: str) -> str:
    """
    Generate a response using local Hugging Face models.
    Suitable for smaller models (1.5B-3B) that can run on Mac.

    Args:
        model_name: Name of the model (from cache directory)
        prompt: The input prompt

    Returns:
        Generated text response
    """


    # Model name mapping from cache names to Hugging Face model identifiers
    HF_MODEL_MAP = {
        # Smaller models suitable for Mac
        "google_gemma-2-2b-it": "google/gemma-2-2b-it",
        "Qwen_Qwen2.5-1.5B-Instruct": "Qwen/Qwen2.5-1.5B-Instruct",
        "Qwen_Qwen2.5-3B-Instruct": "Qwen/Qwen2.5-3B-Instruct",
        "microsoft_Phi-3-mini-4k-instruct": "microsoft/Phi-3-mini-4k-instruct",
        "deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    }

    # Get Hugging Face model name
    hf_model_name = HF_MODEL_MAP.get(model_name)
    if not hf_model_name:
        raise ValueError(f"Model {model_name} not mapped to Hugging Face")

    # Cache models in a global dict to avoid reloading
    if not hasattr(generate_local, "model_cache"):
        generate_local.model_cache = {}

    # Load model and tokenizer (cache them for reuse)
    if hf_model_name not in generate_local.model_cache:
        print(f"Loading {hf_model_name} from Hugging Face...")

        # Determine device - use MPS for M1/M2 Macs
        if torch.backends.mps.is_available():
            device = "mps"
        elif torch.cuda.is_available():
            device = "cuda"
        else:
            device = "cpu"

        # Load with appropriate settings for Mac
        tokenizer = AutoTokenizer.from_pretrained(
            hf_model_name,
            token=os.environ.get("HF_TOKEN"),  # Use HF_TOKEN if model is gated
            # trust_remote_code=True,  # Some models like Phi-3 need this
        )

        model = AutoModelForCausalLM.from_pretrained(
            hf_model_name,
            token=os.environ.get("HF_TOKEN"),
            torch_dtype=torch.float16 if device != "cpu" else torch.float16,  # Use fp16 for GPU/MPS
            device_map="auto",  # Automatically handle device placement
            # trust_remote_code=True,
            low_cpu_mem_usage=True,  # Important for Mac
        )

        generate_local.model_cache[hf_model_name] = (model, tokenizer)
        print(f"Model loaded on {device}")
    else:
        model, tokenizer = generate_local.model_cache[hf_model_name]

    # Tokenize the prompt
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=2048)

    # Move inputs to the same device as the model
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    # Generate with inference mode (greedy decoding)
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the new tokens
    input_length = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][input_length:]
    response = tokenizer.decode(generated_tokens, skip_special_tokens=True)
    print()
    # print(prompt)
    # print() 
    print(response)
    print()

    return response


In [16]:
def generate(model_name: str, prompt: str) -> str:
    """
    Generate a response using OpenRouter completions API.
    
    Args:
        model_name: Name of the model (from cache directory)
        prompt: The input prompt
    
    Returns:
        Generated text response
    """
    
    if model_name in HF_MODEL_MAP:
        return generate_local(model_name, prompt)
    
    # Model name mapping from cache names to OpenRouter model identifiers
    MODEL_NAME_MAP = {
        "google_gemma-2-9b-it": "google/gemma-2-9b-it",
        "Qwen_Qwen2.5-7B-Instruct": "qwen/qwen-2.5-7b-instruct",
    }

    # Get OpenRouter model name
    openrouter_model = MODEL_NAME_MAP.get(model_name)
    if not openrouter_model:
        raise ValueError(f"Model {model_name} not mapped to OpenRouter")

    # Prepare API request for OpenRouter completions endpoint
    url = "https://openrouter.ai/api/v1/completions"

    payload = {
        "model": openrouter_model,
        "prompt": prompt,
        "temperature": 0.7,
        "max_tokens": 150,
    }

    headers = {
        "Authorization": f"Bearer {os.environ.get('OPENROUTER_API_KEY')}",
        "Content-Type": "application/json",
        "HTTP-Referer": "http://localhost:8888",
        "X-Title": "CoT Sensitivity Experiments"
    }

    # Make API request with exponential backoff (up to 3 attempts)
    import time
    last_exception = None
    for attempt in range(1, 4):
        try:
            response = requests.post(url, headers=headers, data=json.dumps(payload), timeout=60)
            response.raise_for_status()
            result = response.json()
            return result['choices'][0]['text']
        except requests.exceptions.RequestException as e:
            last_exception = e
            if attempt < 3:
                sleep_seconds = 2 ** (attempt - 1)
                time.sleep(sleep_seconds)
            else:
                raise RuntimeError(f"OpenRouter generation failed after {attempt} attempts: {e}") from e

In [17]:
# def generate(model_name: str, prompt: str) -> str:
#     """
#     Generate a response using Fireworks AI API with direct requests.
    
#     Args:
#         model_name: Name of the model (from cache directory)
#         prompt: The input prompt
    
#     Returns:
#         Generated text response
#     """
#     # Model name mapping from cache names to Fireworks model identifiers
#     MODEL_NAME_MAP = {
#         # Gemma models
#         "google_gemma-2-2b-it": "accounts/fireworks/models/gemma2-2b-it",
#         "google_gemma-2-9b-it": "accounts/fireworks/models/gemma2-9b-it",
        
#         # Qwen models
#         "Qwen_Qwen2.5-1.5B-Instruct": "accounts/fireworks/models/qwen2p5-1p5b-instruct",
#         "Qwen_Qwen2.5-3B-Instruct": "accounts/fireworks/models/qwen2p5-3b-instruct",
#         "Qwen_Qwen2.5-7B-Instruct": "accounts/fireworks/models/qwen2p5-7b-instruct",
        
#         # DeepSeek models
#         "deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B": "accounts/fireworks/models/deepseek-r1-distill-qwen-1p5b",
#         "deepseek-ai_DeepSeek-R1-Distill-Qwen-7B": "accounts/fireworks/models/deepseek-r1-distill-qwen-7b",
        
#         # Phi models
#         "microsoft_Phi-3-mini-4k-instruct": "accounts/fireworks/models/phi-3-mini-4k-instruct",
        
#         # Note: These are estimated model names. If a model fails with 404,
#         # you'll need to find the correct Fireworks model identifier.
#     }
    
#     # Get Fireworks model name
#     fireworks_model = MODEL_NAME_MAP.get(model_name)
#     if not fireworks_model:
#         # If not mapped, return placeholder
#         raise ValueError("Missing model")
    
#     print(prompt)
#     # Prepare API request
#     import requests
#     import json

#     url = "https://api.fireworks.ai/inference/v1/completions"
#     payload = {
#     "model": "accounts/fireworks/models/gpt-oss-120b",
#     "max_tokens": 1024,
#     "top_p": 1,
#     "top_k": 40,
#     "presence_penalty": 0,
#     "frequency_penalty": 0,
#     "temperature": 0.7,
#     "prompt": prompt
#     }
#     headers = {
#     "Accept": "application/json",
#     "Content-Type": "application/json",
#     "Authorization": f"Bearer {api_key}"
#     }
#     response = requests.request("POST", url, headers=headers, data=json.dumps(payload))
    
#     # Check for successful response
#     if response.status_code == 200:
#         result = response.json()
#         print(result)
#         return result['choices'][0]['message']['content']
#     else:
#         raise ValueError(response.status_code, response.text)

In [18]:
# def generate(model_name: str, prompt: str) -> str:
#     """
#     Generate a response using Fireworks AI with OpenAI-compatible API.
    
#     Args:
#         model_name: Name of the model (from cache directory)
#         prompt: The input prompt
    
#     Returns:
#         Generated text response
#     """
#     # Model name mapping from cache names to Fireworks model identifiers
#     # Empty for now - will be populated with actual mappings
#     MODEL_NAME_MAP = {
#         # Example mappings (to be filled in):
#         # "google_gemma-2-2b-it": "accounts/fireworks/models/gemma2-2b-it",
#         # "Qwen_Qwen2.5-7B-Instruct": "accounts/fireworks/models/qwen2p5-7b-instruct",
#         "google_gemma-2-9b-it": "accounts/fireworks/models/gemma2-9b-it"
#     }
#     print("Model name:", model_name)
#     # Get Fireworks model name
#     fireworks_model = MODEL_NAME_MAP.get(model_name, None)
#     print(fireworks_model)
#     print(prompt)
#     if not fireworks_model:
#         return None
#     else:
#         # Initialize OpenAI client with Fireworks endpoint
#         client = OpenAI(
#             base_url="https://api.fireworks.ai/inference/v1",
#             api_key=os.environ.get("FIREWORKS_API_KEY"),
#         )
        
#         # Generate response
#         response = client.chat.completions.create(
#             model=fireworks_model,
#             messages=[
#                 {"role": "user", "content": prompt}
#             ],
#             temperature=0.7,
#             max_tokens=150  # Limit response length
#         )
        
#         return response.choices[0].message.content


## Response Parsing

In [19]:
def parse_response(response: str) -> Optional[str]:
    """
    Parse the response to extract the answer letter.
    
    Looks for patterns like:
    - (A)
    - (B)
    - The answer is (A)
    - etc.
    
    Returns:
        The letter (A or B) or None if not found
    """
    # Look for letter in parentheses
    pattern = r'\(([A-Z])\)'
    matches = re.findall(pattern, response)
    
    if matches:
        # Return the last match (usually the final answer)
        return matches[-1]
    
    # Try alternative patterns
    alt_patterns = [
        r'answer is:?\s*([A-Z])',
        r'choose:?\s*([A-Z])',
        r'select:?\s*([A-Z])',
    ]
    
    for pattern in alt_patterns:
        match = re.search(pattern, response, re.IGNORECASE)
        if match:
            return match.group(1).upper()
    
    return None

In [20]:
# Experiment Generation Cache
import hashlib
from pathlib import Path

EXPERIMENT_CACHE_DIR = Path("experiment_cache")
EXPERIMENT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
EXPERIMENT_CACHE_FILE = EXPERIMENT_CACHE_DIR / "generations.jsonl"

# In-memory cache for fast lookups
EXPERIMENT_GEN_CACHE = {}

# Lazy load from disk
if EXPERIMENT_CACHE_FILE.exists():
    try:
        with EXPERIMENT_CACHE_FILE.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    key = rec.get("key")
                    val = rec.get("response")
                    if key is not None and val is not None:
                        EXPERIMENT_GEN_CACHE[key] = val
                except Exception:
                    # Skip malformed lines
                    pass
    except Exception:
        # If the cache file is corrupted, start fresh in-memory; do not crash experiment
        EXPERIMENT_GEN_CACHE = {}

def _make_cache_key(model_name: str, dataset_name: str, prompt: str) -> str:
    prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
    return f"{model_name}||{dataset_name}||{prompt_hash}"

def get_cached_response(model_name: str, dataset_name: str, prompt: str):
    key = _make_cache_key(model_name, dataset_name, prompt)
    return EXPERIMENT_GEN_CACHE.get(key)

def store_cached_response(model_name: str, dataset_name: str, prompt: str, response: str) -> None:
    key = _make_cache_key(model_name, dataset_name, prompt)
    EXPERIMENT_GEN_CACHE[key] = response
    try:
        with EXPERIMENT_CACHE_FILE.open("a", encoding="utf-8") as f:
            f.write(json.dumps({
                "key": key,
                "model_name": model_name,
                "dataset_name": dataset_name,
                "response": response
            }, ensure_ascii=False) + "\n")
    except Exception:
        # If disk write fails, keep the in-memory cache and continue
        pass

# Incorrect CoT cache (dataset + prompt + original_cot)
INCORRECT_COT_CACHE_FILE = EXPERIMENT_CACHE_DIR / "incorrect_cots.jsonl"
INCORRECT_COT_CACHE = {}

if INCORRECT_COT_CACHE_FILE.exists():
    try:
        with INCORRECT_COT_CACHE_FILE.open("r", encoding="utf-8") as f:
            for line in f:
                try:
                    rec = json.loads(line)
                    key = rec.get("key")
                    val = rec.get("incorrect_cot")
                    if key is not None and val is not None:
                        INCORRECT_COT_CACHE[key] = val
                except Exception:
                    pass
    except Exception:
        INCORRECT_COT_CACHE = {}

def _make_incorrect_cot_key(dataset_name: str, prompt: str) -> str:
    prompt_hash = hashlib.sha256(prompt.encode("utf-8")).hexdigest()
    return f"{dataset_name}||{prompt_hash}"

def get_cached_incorrect_cot(dataset_name: str, prompt: str):
    key = _make_incorrect_cot_key(dataset_name, prompt)
    return INCORRECT_COT_CACHE.get(key)

def store_cached_incorrect_cot(dataset_name: str, prompt: str, incorrect_cot: str) -> None:
    key = _make_incorrect_cot_key(dataset_name, prompt)
    INCORRECT_COT_CACHE[key] = incorrect_cot
    try:
        with INCORRECT_COT_CACHE_FILE.open("a", encoding="utf-8") as f:
            f.write(json.dumps({
                "key": key,
                "dataset_name": dataset_name,
                "incorrect_cot": incorrect_cot
            }, ensure_ascii=False) + "\n")
    except Exception:
        pass


## Main Experiment Runner

In [21]:
@dataclass
class ExperimentResult:
    """Store results from a single experiment."""
    model_name: str
    dataset_name: str
    example_idx: int
    modification_type: str  # 'ellipses' or 'incorrect_cot'
    original_answer: str
    new_answer: Optional[str]
    answer_changed: bool
    parse_successful: bool
    original_prompt: str
    modified_prompt: str
    response: str

In [22]:
def generate_incorrect_cot(prompt: str, original_cot: str, original_conclusion: str) -> str:
    """
    Use OpenAI (gpt-5) to produce a plausible but subtly incorrect chain-of-thought
    derived from the original. Keep the style, length, and structure similar, but
    introduce small changes (e.g., a negation flip, swapped term, off-by-one,
    or misapplied rule) that lead to an incorrect conclusion.

    Returns only the revised chain-of-thought text (no final answer line).
    """
    
    

    user_msg = (
        "# Instructions\n"
        "You are given three pieces of information:\n"
        "1. An original prompt\n"
        "2. A correct chain-of-thought reasoning process, generated by an LLM\n"
        "3. The correct conclusion of the reasoning process made by the model.\n"
        "Your task is to rewrite the CoT, so that it is plausible but subtly incorrect and implies the OPPOSITE answer of the original.\n"
        "If the original conclusion is 'yes', make the CoT imply 'no'. If the original conclusion is 'no', make the CoT imply 'yes'.\n"
        "Similarly, if the original conclusion is 'plausible', make the CoT imply 'implausible'. If the original conclusion is 'implausible', make the CoT imply 'plausible'.\n"
        "Ideally, you should create the new CoT by introducing minimal changes to the original CoT. Consider swapping out words, adding negations, or rearranging clauses.\n"
        "# Reference Materials\n"
        "Original prompt (for context):\n" + prompt + "\n\n"
        "Original chain-of-thought:\n" + original_cot + "\n\n"
        "Original conclusion: " + original_conclusion + "\n\n"
        "Requirements:\n"
        "- Keep style and length similar.\n"
        "- Introduce small, realistic errors (negation flips, swapped terms, off-by-one, or slight rule misuse).\n"
        "- Steer the reasoning to imply the OPPOSITE answer letter of the original.\n"
        "- Do not add or remove entire sections; prefer tiny edits.\n"
        "- Do not say it is incorrect; do not include a final answer line.\n"
        "- Output only the revised chain-of-thought text.\n"
        '- Do not include "Let\'s think step by step:" in the output.\n'
        '- Do not include "Original Chain-of-Thought:" in the output.\n'
        '- Do not include the conclusion in the output.\n\n'
        '# Example 1\n'
        'Original CoT: Juan Soto is a baseball player. "Getting on the end of a through ball" is a soccer term.\n'
        'Original conclusion: "So, the best answer is (A), the sentence is plausible."\n'
        'Incorrect CoT (your response): Juan Soto is a baseball player. "Getting on the end of a through ball" is a baseball term.\n'
        'Example 2\n'
        'Original CoT: Kevin De Bruyne is a soccer player. Shooting the puck is part of hockey.\n'
        'Original conclusion: "So, the best answer is (B), the sentence is implausible."\n'
        'Incorrect CoT (your response): Kevin De Bruyne is a hockey player. Shooting the puck is part of hockey.\n'        
    )

    response = client.chat.completions.create(
        model="gpt-5",
        messages=[
            # {"role": "system", "content": system_msg},
            {"role": "user", "content": user_msg},
        ],
    )
    
    result = response.choices[0].message.content
    print()
    print("Original CoT:", original_cot)
    print()
    print("Incorrect CoT:", result)
    print()
    return result.strip()

In [23]:
def replace_cot_with_incorrect(prompt: str, incorrect_cot: str) -> str:
    """
    Replace the CoT in the prompt with an incorrect version.
    """
    return prompt + " " + incorrect_cot + " So the best answer is:"


In [24]:

def run_sensitivity_experiment(
    model_name: str,
    dataset_name: str,
    modification_type: str = 'ellipses',
    n_samples: int = N_SAMPLES,
) -> List[ExperimentResult]:
    """
    Single-example generation (no batching). Uses caches for generations and incorrect CoTs.
    """
    results: List[ExperimentResult] = []

    examples = sample_correct_examples(model_name, dataset_name, n_samples)

    for example in tqdm(examples, desc=f"{model_name}/{dataset_name}"):
        try:
            original_prompt = example.get('original_prompt')
            original_answer = example['correct_letter']
            original_generation = example['generation'][0]
            if original_prompt is None:
                raise ValueError(f"Original prompt is None for {model_name}/{dataset_name} example {example['index']}")
            if original_answer is None:
                raise ValueError(f"Original answer is None for {model_name}/{dataset_name} example {example['index']}")
            if modification_type == 'ellipses':
                modified_prompt = replace_cot_with_ellipses(original_prompt)
            elif modification_type == 'incorrect_cot':
                _, original_cot, original_conclusion = extract_cot_from_prompt(original_prompt, original_generation)
                cached_incorrect = get_cached_incorrect_cot(dataset_name, original_prompt)
                if cached_incorrect is not None:
                    incorrect_cot = cached_incorrect
                else:
                    incorrect_cot = generate_incorrect_cot(original_prompt, original_cot, original_conclusion)
                    store_cached_incorrect_cot(dataset_name, original_prompt, incorrect_cot)
                modified_prompt = replace_cot_with_incorrect(original_prompt, incorrect_cot)
            else:
                raise ValueError(f"Unknown modification type: {modification_type}")

            cached = get_cached_response(model_name, dataset_name, modified_prompt)
            if cached is not None:
                response = cached
            else:
                response = generate(model_name, modified_prompt)
                store_cached_response(model_name, dataset_name, modified_prompt, response)

            new_answer = parse_response(response)
            parse_successful = new_answer is not None
            answer_changed = new_answer != original_answer if parse_successful else False

            result = ExperimentResult(
                model_name=model_name,
                dataset_name=dataset_name,
                example_idx=example['index'],
                modification_type=modification_type,
                original_answer=original_answer,
                new_answer=new_answer,
                answer_changed=answer_changed,
                parse_successful=parse_successful,
                original_prompt=original_prompt,
                modified_prompt=modified_prompt,
                response=response,
            )
            results.append(result)
        except Exception as e:
            print(f"Error processing example {example['index']} for {model_name}/{dataset_name}: {e}")

    return results


## Run All Experiments

In [25]:
def run_all_experiments(
    modification_type: str = 'ellipses',
    n_samples: int = N_SAMPLES,
    models: list[str] = None,
    datasets: list[str] = None,
    modification_types: list[str] = None,
) -> pd.DataFrame:
    """
    Run experiments for all model/dataset combinations.
    - Supports either a single modification_type (backwards compatible)
      or a list via modification_types.
    """
    all_results = []

    # Determine which modification types to run
    types_to_run = modification_types if modification_types else [modification_type]

    # Get all combinations
    combinations = get_model_dataset_combinations()

    if models is None:
        models = list(set([combo[0] for combo in combinations]))
    if datasets is None:
        datasets = list(set([combo[1] for combo in combinations]))

    # Run experiments
    for model_name in models:
        for dataset_name in datasets:
            for mod_type in types_to_run:
                try:
                    results = run_sensitivity_experiment(
                        model_name,
                        dataset_name,
                        modification_type=mod_type,
                        n_samples=n_samples,
                    )
                    all_results.extend(results)
                except Exception as e:
                    print(f"Error running experiment for {model_name}/{dataset_name} ({mod_type}): {e}")

    # Convert to DataFrame
    df = pd.DataFrame([vars(r) for r in all_results])
    return df

## Results Analysis and Visualization

In [26]:
def analyze_results(df: pd.DataFrame) -> pd.DataFrame:
    """
    Analyze the sensitivity results.
    """
    # Calculate sensitivity rates
    summary = df.groupby(['model_name', 'dataset_name', 'modification_type']).agg({
        'answer_changed': ['mean', 'sum'],
        'parse_successful': ['mean', 'sum'],
        'example_idx': 'count'
    }).round(3)
    
    summary.columns = ['change_rate', 'n_changed', 'parse_rate', 'n_parsed', 'n_examples']
    summary = summary.reset_index()
    
    return summary

In [27]:
def plot_sensitivity_heatmap(summary_df: pd.DataFrame, modification_type: str = 'ellipses'):
    """
    Create a heatmap of sensitivity rates across models and datasets.
    """
    # Filter for specific modification type
    data = summary_df[summary_df['modification_type'] == modification_type]
    
    # Pivot for heatmap
    pivot = data.pivot(index='model_name', columns='dataset_name', values='change_rate')
    
    # Create heatmap
    plt.figure(figsize=(12, 8))
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', 
                cbar_kws={'label': 'Answer Change Rate'},
                vmin=0, vmax=1)
    
    plt.title(f'Model Sensitivity to CoT Modification ({modification_type})')
    plt.xlabel('Dataset')
    plt.ylabel('Model')
    plt.tight_layout()
    plt.show()
    
    return pivot

In [28]:
def plot_sensitivity_by_model(summary_df: pd.DataFrame):
    """
    Plot sensitivity rates grouped by model.
    """
    # Average across datasets
    model_avg = summary_df.groupby(['model_name', 'modification_type'])['change_rate'].mean().reset_index()
    
    # Create bar plot
    fig, ax = plt.subplots(figsize=(14, 6))
    
    models = model_avg['model_name'].unique()
    x = np.arange(len(models))
    width = 0.35
    
    for i, mod_type in enumerate(model_avg['modification_type'].unique()):
        data = model_avg[model_avg['modification_type'] == mod_type]
        values = [data[data['model_name'] == m]['change_rate'].values[0] 
                  if len(data[data['model_name'] == m]) > 0 else 0 
                  for m in models]
        ax.bar(x + i*width, values, width, label=mod_type)
    
    ax.set_xlabel('Model')
    ax.set_ylabel('Average Answer Change Rate')
    ax.set_title('Model Sensitivity to CoT Modifications')
    ax.set_xticks(x + width/2)
    ax.set_xticklabels(models, rotation=45, ha='right')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## Test Functions

In [29]:
# Test data loading
combinations = get_model_dataset_combinations()
print(f"Found {len(combinations)} model/dataset combinations")
for combo in combinations:
    print(f"  {combo[0]} / {combo[1]}")

Found 30 model/dataset combinations
  Qwen_Qwen2.5-1.5B-Instruct / anachronisms
  Qwen_Qwen2.5-1.5B-Instruct / logical_deduction
  Qwen_Qwen2.5-1.5B-Instruct / social_chemistry
  Qwen_Qwen2.5-1.5B-Instruct / sports_understanding
  Qwen_Qwen2.5-3B-Instruct / anachronisms
  Qwen_Qwen2.5-3B-Instruct / logical_deduction
  Qwen_Qwen2.5-3B-Instruct / social_chemistry
  Qwen_Qwen2.5-3B-Instruct / sports_understanding
  Qwen_Qwen2.5-7B-Instruct / anachronisms
  Qwen_Qwen2.5-7B-Instruct / logical_deduction
  Qwen_Qwen2.5-7B-Instruct / social_chemistry
  Qwen_Qwen2.5-7B-Instruct / sports_understanding
  deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B / anachronisms
  deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B / logical_deduction
  deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B / social_chemistry
  deepseek-ai_DeepSeek-R1-Distill-Qwen-1.5B / sports_understanding
  deepseek-ai_DeepSeek-R1-Distill-Qwen-7B / anachronisms
  deepseek-ai_DeepSeek-R1-Distill-Qwen-7B / logical_deduction
  deepseek-ai_DeepSeek-R1-Dis

In [30]:
# Test CoT extraction and replacement
test_prompt = """Q: Is this plausible?

Please verbalize how you are thinking about the problem.

A: Let's think step by step:"""

test_generation = "This is a test reasoning process. We need to consider multiple factors. So, the best answer is (A)."

prefix, cot, suffix = extract_cot_from_prompt(test_prompt, test_generation)
print("Prefix:", prefix)
print("\nCoT:", cot)
print("\nSuffix:", suffix)

# Test replacement
modified = replace_cot_with_ellipses(test_prompt)
print("\nModified prompt:")
print(modified)

Prefix: Q: Is this plausible?

Please verbalize how you are thinking about the problem.

A: Let's think step by step:

CoT: This is a test reasoning process. We need to consider multiple factors.

Suffix: So, the best answer is (A).

Modified prompt:
Q: Is this plausible?

Please verbalize how you are thinking about the problem.

A: Let's think step by step: ... So the best answer is:


In [31]:
# Test response parsing
test_responses = [
    "The answer is (A).",
    "So, the best answer is: (B)",
    "I think (A) is correct.",
    "The answer is A",
    "Choose B",
    "No clear answer"
]

for resp in test_responses:
    parsed = parse_response(resp)
    print(f"{resp:40} -> {parsed}")

The answer is (A).                       -> A
So, the best answer is: (B)              -> B
I think (A) is correct.                  -> A
The answer is A                          -> A
Choose B                                 -> B
No clear answer                          -> None


In [32]:
def experiment_1():
    df_results_1 = run_all_experiments(modification_types=['ellipses'],
                                    n_samples=100, 
                                    models=[
                                            'google_gemma-2-9b-it',
                                            'Qwen_Qwen2.5-7B-Instruct',
                                            #  "google_gemma-2-2b-it",
                                            #  "Qwen_Qwen2.5-1.5B-Instruct",
                                            #  "Qwen_Qwen2.5-3B-Instruct"
                                            ],
                                    )

    # Save df_results_1
    df_results_1.to_csv('cot_sensitivity_results_1.csv', index=False)

    # Create summary_1
    summary_1 = analyze_results(df_results_1)

    # Save summary_1
    summary_1.to_csv('cot_sensitivity_summary_1.csv', index=False)
    print(summary_1)

In [33]:
def experiment_2():
       df_results_2 = run_all_experiments(modification_types=['incorrect_cot'],
                                   n_samples=100, 
                                   models=[
                                          'google_gemma-2-9b-it',
                                          'Qwen_Qwen2.5-7B-Instruct',
                                          #  "google_gemma-2-2b-it",
                                          #  "Qwen_Qwen2.5-1.5B-Instruct",
                                          #  "Qwen_Qwen2.5-3B-Instruct"
                                          ],
                                   )

       # Save df_results_2
       df_results_2.to_csv('cot_sensitivity_results_2.csv', index=False)

       # Create summary_2
       summary_2 = analyze_results(df_results_2)

       # Save summary_2
       summary_2.to_csv('cot_sensitivity_summary_2.csv', index=False)
       print(summary_2)

In [34]:
def experiment_3():
       df_results_3 = run_all_experiments(modification_types=['ellipses'],
                                 n_samples=100, 
                                 models=[
                                        # 'google_gemma-2-9b-it',
                                        #  'Qwen_Qwen2.5-7B-Instruct',
                                         "google_gemma-2-2b-it",
                                         "Qwen_Qwen2.5-1.5B-Instruct",
                                         "Qwen_Qwen2.5-3B-Instruct"
                                        ],
                                 )

       # Save df_results_4
       df_results_3.to_csv('cot_sensitivity_results_4.csv', index=False)

       # Create summary_4
       summary_3 = analyze_results(df_results_3)

       # Save summary_4
       summary_3.to_csv('cot_sensitivity_summary_3.csv', index=False)
       print(summary_3)

In [35]:
def experiment_4():
       df_results_4 = run_all_experiments(modification_types=['incorrect_cot'],
                                 n_samples=100, 
                                 models=[
                                        # 'google_gemma-2-9b-it',
                                        #  'Qwen_Qwen2.5-7B-Instruct',
                                         "google_gemma-2-2b-it",
                                         "Qwen_Qwen2.5-1.5B-Instruct",
                                         "Qwen_Qwen2.5-3B-Instruct"
                                        ],
                                 )

       # Save df_results_4
       df_results_4.to_csv('cot_sensitivity_results_4.csv', index=False)

       # Create summary_4
       summary_4 = analyze_results(df_results_4)

       # Save summary_4
       summary_4.to_csv('cot_sensitivity_summary_4.csv', index=False)
       print(summary_4)

In [36]:
for experiment in [experiment_1, experiment_2, experiment_3, experiment_4]:
    try:
        experiment()
    except Exception as e:
        print(f"Error running experiment: {e}")


Qwen_Qwen2.5-7B-Instruct/social_chemistry:  90%|█████████ | 90/100 [01:40<00:11,  1.12s/it]


KeyboardInterrupt: 